In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time
from datetime import datetime
from pyspark.sql.window import Window
import json
import random

In [2]:
def create_spark_session():
    spark=SparkSession.builder.appName('Dataset').getOrCreate()
    return spark    

In [3]:
def create_datastreamReader():
    userSchema = StructType().add("Acc No", "integer").add("Transaction", "string").add("Time Stamp", "string").add("Amount", "integer")
    streaming_df = spark.readStream.schema(userSchema) \
        .option('maxFilesPerTrigger', 1)\
        .json('./data')

    accountDF = streaming_df.withColumn('Times', to_timestamp(col('Time Stamp'))).drop('Time Stamp')
    spark.conf.set("spark.sql.shuffle.partitions", 1)

    return accountDF

In [4]:
def get_aggregate_count_lessthan5L(accountDF):
    lessthan_five_lakh = accountDF.select('Acc No', 'Amount', 'Times').where(col('Amount') < 500000).groupBy('Acc No').count()
    lessquery = lessthan_five_lakh.writeStream.queryName('Min_Amount_complete')\
        .trigger(processingTime ="20 seconds")\
        .format('console')\
        .outputMode('complete')\
        .start()
    lessquery.awaitTermination()

In [5]:
def get_trans_details_perTrigger():
    lessthan_five_lakh = accountDF.select('Acc No', 'Amount', 'Times').where(col('Amount') < 500000)
    append_query = lessthan_five_lakh.writeStream.queryName('Min_Amount_append')\
        .trigger(processingTime ="20 seconds")\
        .format('console')\
        .outputMode('append')\
        .start()
    append_query.awaitTermination()


In [6]:
# Count of records which satisfy the given condition for each trigger
def transactionFilter_withWaterMark():
    windowedCounts = (
        accountDF.withColumn("Times", to_timestamp(col("Times")))\
        .withWatermark("Times", "1 minute")\
        .where(col("Amount") < 500000)\
        .groupBy("Acc No", window("Times", "20 seconds"))\
        .count()
    )

    query = (
        windowedCounts.writeStream.outputMode("append")\
        .format("console")\
        .option("truncate", "false")\
        .trigger(processingTime="20 seconds")\
        .start()
    )

In [ ]:
spark = create_spark_session()
accountDF = create_datastreamReader()
get_aggregate_count_lessthan5L(accountDF)
#get_trans_details_perTrigger()
#transactionFilter_withWaterMark()